# CP Postnatal (mouse) Removing doublets after subclustering and final plots

In [ ]:
import numpy as np
np.random.default_rng(2159)

PYTHONHASHSEED=0

In [ ]:
#load packages
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import os

sc.settings.verbosity = 3   
sc.set_figure_params(scanpy=True, fontsize=12, dpi_save=180, format='svg', transparent=True)
sc.settings.set_figure_params(dpi=80)

%matplotlib inline

image_format = 'svg' # e.g .png, .svg, etc.

In [ ]:
my_c = ['#fe864a', '#c9d179', '#95d0fc', '#fbdd7e', '#388004', '#c760ff', '#76cd26', '#ffbacd', '#d6b4fc',
        '#cfaf7b','#8b88f8', '#acc2d9', '#894585', '#82cbb2', '#d99b82', '#fdee73', '#5170d7', '#ffd8b1', 
        '#b7e1a1', '#82a67d', '#b7fffa', '#b790d4']

# Load data

In [ ]:
adata_raw = sc.read_h5ad('Postnatal_unfiltered.h5ad')

In [ ]:
adata_annotated = sc.read_h5ad('Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
epithelial = sc.read_h5ad('noDBL_Epithelial_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
endothelial =sc.read_h5ad('noDBL_Endothelial_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
mesenchymal = sc.read_h5ad('noDBL_Mesenchymal_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
immune = sc.read_h5ad('noDBL_Immune_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
neurons_glia = sc.read_h5ad('Postnatal_scrnaseq_Neuronal_and_Glial_cells.h5ad')

In [ ]:
adata_concat = epithelial.concatenate(endothelial, mesenchymal, immune, neurons_glia)

In [ ]:
adata_concat

In [ ]:
adata_concat.obs_names = [cell[0:-2] for cell in adata_concat.obs_names]

In [ ]:
cells = [cell for cell in adata_concat.obs_names]

In [ ]:
adata_filt = adata_raw[cells, :]

In [ ]:
adata_filt

In [ ]:
adata_filt.obs['Sub_cell_type'] = False

In [ ]:
for cell in adata_concat.obs_names:
    adata_filt.obs['Sub_cell_type'].loc[cell] = adata_concat.obs['Cell_type'].loc[cell]


In [ ]:
adata_annotated = adata_annotated[cells, :]

In [ ]:
adata_annotated

In [ ]:
adata_filt.obs['Cell_type'] = False

In [ ]:
for cell in adata_annotated.obs_names:
    adata_filt.obs['Cell_type'].loc[cell] = adata_annotated.obs['Cell_type'].loc[cell]

In [ ]:
adata_filt.obsm = adata_annotated.obsm

In [ ]:
import colorsys
N = 37
HSV_tuples = [(x*1.0/N, 0.7, 0.4) for x in range(N)]
RGB_tuples = map(lambda x: colorsys.hsv_to_rgb(*x), HSV_tuples)

In [ ]:
import random
color = []
for j in range(50):
    rand_colors = ["#"+''.join([random.choice('ABCDEF0123456789') for i in range(6)])]
    color.append(rand_colors[0])

print(color)

In [ ]:
sc.pl.umap(adata_filt, color=['Cell_type', 'Sub_cell_type', 'Age'], size=1, ncols=2, wspace=0.7, palette=my_c)

In [ ]:
adata = adata_filt.copy()

In [ ]:
adata.write_h5ad('noDBL_Postnatal_noDBL_raw_annotated.h5ad')

In [ ]:
## normalize to depth 10 000
sc.pp.normalize_per_cell(adata,counts_per_cell_after=1e4)

## logaritmize
sc.pp.log1p(adata)

In [ ]:
adata.raw = adata.copy()

In [ ]:
adata

In [ ]:
sc.pl.violin(adata, ['total_counts','n_genes_by_counts','pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb'], 
             groupby="batch", rotation=90, stripplot=False, palette=my_c, save='_noDBL_afterfilter_metrics.svg')

In [ ]:
sc.tl.rank_genes_groups(adata, 'Cell_type', method='wilcoxon', key_added = "wilcoxon_Cell_type", use_raw=True)

In [ ]:
adata.obs['Cell_type'].unique()

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_Cell_type" , groupby="Cell_type", cmap="YlGnBu", vmin=0, vmax=10,
                               save='_noDBL_nofiltered_all.svg')

In [ ]:
sc.pl.umap(adata, color=['Cell_type', 'Age'], size=1, wspace=0.7, palette=my_c)

In [ ]:
adata.obs['P2/P3'] = False
adata.obs['P11'] = False
adata.obs['P60'] = False

In [ ]:
ages = ['P2/P3', 'P11', 'P60']

In [ ]:
for p in ages:
    for cell in adata.obs_names:
        if adata.obs['Age'].loc[cell] == p:
            adata.obs[p].loc[cell] = p
        else:
            adata.obs[p].loc[cell] = np.nan

In [ ]:
sc.pl.umap(adata, color=ages, size=1, wspace=0.5, na_color="#FF000000", save='_noDBL_ages_all.svg')

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="Cell_type", method='wilcoxon', n_genes=50)

clust_df = pd.DataFrame(adata.uns['rank_genes_groups']['names'])
clust_df.to_csv("noDBL_Top_50_Postnatal_mouse_marker_genes_CP.csv")

In [ ]:
markers = ['Flt1', 'Plvap', 'Pecam1', 'Ly6a',
            'Folr1', 'Clic6', 'Otx2', 'Krt18',
            'C1qa', 'C1qb', 'Fcer1g', 'C1qc',
            'Dcn', 'Itga8', 'Col3a1', 'Lum',
            'Sox6', 'Ptprz1', 'Mbp', 'Plp1',]

In [ ]:
sns.set_style({'axes.facecolor': 'white',
 'axes.edgecolor': '.8',
 'axes.grid': False,
 'axes.axisbelow': 'line',
 'axes.labelcolor': 'black',
 'figure.facecolor': 'white',
 'grid.color': '.8',
 'grid.linestyle': '-',
 'text.color': 'black',
 'xtick.color': 'black',
 'ytick.color': 'black',
 'xtick.direction': 'out',
 'ytick.direction': 'out',
 'patch.edgecolor': 'black',
 'patch.force_edgecolor': False,
 'image.cmap': 'viridis',
 'font.family': ['sans-serif'],
 'font.sans-serif': ['DejaVu Sans',
  'Bitstream Vera Sans',
  'Computer Modern Sans Serif',
  'Lucida Grande',
  'Verdana',
  'Geneva',
  'Lucid',
  'Arial',
  'Helvetica',
  'Avant Garde',
  'sans-serif'],
 'xtick.bottom': True,
 'xtick.top': False,
 'ytick.left': True,
 'ytick.right': False,
 'axes.spines.left': True,
 'axes.spines.bottom': True,
 'axes.spines.right': True,
 'axes.spines.top': True}
)

vp = sc.pl.rank_genes_groups_stacked_violin(adata, var_names=markers, groupby='Cell_type', key='wilcoxon_Cell_type', row_palette=my_c, dendrogram=False, 
                                       yticklabels = True, return_fig=True, figsize=(12,4), save='_NEW_noDBL_celltype_markers_all.svg')

vp.style(ylim=(0,6)).show()

vp.savefig('stacked_violin_NEW_noDBL_celltype_markers_all.svg', format='svg', dpi=180)
axes_dict = vp.get_axes()
print(axes_dict)

In [ ]:
sc.pl.violin(adata, ['total_counts','n_genes_by_counts','pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb'], 
             groupby="Cell_type", rotation=90, stripplot=False, palette=my_c, save='_noDBL_celltypes_metrics.svg')

In [ ]:
def batch_pct_cellType(adata_file,groupby_key,legend_key,pos):
    tmp = pd.crosstab(adata_file.obs[groupby_key],adata_file.obs[legend_key], normalize='index')   
    tmp.plot.bar(stacked=True,rot= 90, grid=False, color=my_c,
                 xlabel='',ax=pos).legend(bbox_to_anchor=[1.20,1])

In [ ]:
fig, (ax1,ax2,ax3) = plt.subplots(3, 1, constrained_layout=True,figsize=(10,15))
batch_pct_cellType(adata,'Cell_type','Age',ax1)
batch_pct_cellType(adata,'Cell_type','batch',ax2)
batch_pct_cellType(adata,'Age','Cell_type',ax3)

fig.savefig('barplots_noDBL_celltypes_age_all.svg', format='svg')

# Cell proportions

In [ ]:
ct = pd.crosstab(adata.obs['batch'],adata.obs['Cell_type'], normalize='index')

In [ ]:
ct

In [ ]:
ct['Age'] = None

In [ ]:
for b in ct.index:
    if 'P2' in b:
        ct['Age'].loc[b] = 'P2/P3'  
    elif 'P3' in b:
        ct['Age'].loc[b] = 'P2/P3'
    elif 'P11' in b:
        ct['Age'].loc[b] = 'P11'
    else:# 'P60' in b:
        ct['Age'].loc[b] = 'P60'

In [ ]:
ct

In [ ]:
ctt = ct.reset_index().melt(id_vars=['batch'])[:-8]

In [ ]:
ctt.rename({'value': 'Percentage of cells'}, axis=1, inplace=True)

In [ ]:
ctt['Age'] = None

In [ ]:
ctt = ctt.set_index(['batch'])

In [ ]:
ctt.index

In [ ]:
for b in ctt.index:
    if 'P2' in b:
        ctt['Age'].loc[b] = 'P2/P3'  
    elif 'P3' in b:
        ctt['Age'].loc[b] = 'P2/P3'
    elif 'P11' in b:
        ctt['Age'].loc[b] = 'P11'
    else:# 'P60' in b:
        ctt['Age'].loc[b] = 'P60'

In [ ]:
ctt

In [ ]:
from matplotlib.legend_handler import HandlerTuple

fig = plt.figure(figsize=(22,10))
ax = fig.add_subplot(111)    # The big subplot
ax1 = fig.add_subplot(211)
ax2 = fig.add_subplot(212)

# Turn off axis lines and ticks of the big subplot
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_color('none')
ax.spines['left'].set_color('none')
ax.spines['right'].set_color('none')
ax.tick_params(labelcolor='w', top=False, bottom=False, left=False, right=False)
ax.grid(False)

sns.boxplot(x = ctt['Cell_type'], y = ctt['Percentage of cells'], data = ctt, hue=ctt['Age'], 
            palette = my_c, showfliers = True, ax=ax1)

sns.stripplot(data=ctt, x='Cell_type', y="Percentage of cells", hue="Age", 
              dodge=True, palette=my_c[6:9], jitter=0.35, ax=ax1)

sns.boxplot(x = ctt['Cell_type'], y = ctt['Percentage of cells'], data = ctt, hue=ctt['Age'], 
            palette = my_c, showfliers = True, ax=ax2)

sns.stripplot(data=ctt, x='Cell_type', y="Percentage of cells", hue="Age", 
              dodge=True, palette=my_c[6:9], jitter=0.355, ax=ax2)


ax1.set_ylim(0.7,1)   # those limits are fake
ax2.set_ylim(0,0.1)
sns.despine(ax=ax2)
sns.despine(ax=ax1, bottom=True)

d = .005  # how big to make the diagonal lines in axes coordinates
kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
ax1.plot((-d, +d), (-d, +d), **kwargs)        # top-left diagonal

kwargs.update(transform=ax2.transAxes)  # switch to the bottom axes
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  # bottom-left diagonal

ax2.legend_.remove()
ax1.set(xlabel=None, ylabel=None)
ax2.set(xlabel=None,ylabel=None)
# ax1.legend_.remove()
ax1.set_xticks([])
ax1.set_yticks(np.arange(0.70,1, 0.1))
ax2.set_yticks(np.arange(0,0.11, 0.05))

from textwrap import wrap
names = [cell for cell in ctt['Cell_type'].unique()]

names = [ '\n'.join(wrap(l, 20)) for l in names ]

handles, labels = ax1.get_legend_handles_labels()
ax1.legend(handles=[(handles[0], handles[3]), (handles[1], handles[4]), (handles[2], handles[5])],
          labels=['P2/P3', 'P11', 'P60'], handlelength=2,
          handler_map={tuple: HandlerTuple(ndivide=None)},bbox_to_anchor=[1.12,1.05],fontsize=20)

ax2.set_xticklabels(names, fontsize=22)
# ax2.tick_params(names, labelsize=18)
ax1.tick_params(axis='y', labelsize=20)
ax2.tick_params(axis='y', labelsize=20)
ax.set_ylabel('Percantage of cells', fontsize=22)
ax.yaxis.set_label_coords(-0.05, 0.5)

plt
plt.show()

fig.savefig('boxplots_noDBL_all_celltypes_all.svg')

## Statistics on cell proportions

In [ ]:
ct_mann_whitney_u = ct.copy()

In [ ]:
ct_mann_whitney_u['Age'] = None

In [ ]:
for b in ct_mann_whitney_u.index:
    if 'P2' in b:
        ct_mann_whitney_u['Age'].loc[b] = 'P2/P3/P11'  
    elif 'P3' in b:
        ct_mann_whitney_u['Age'].loc[b] = 'P2/P3/P11'
    elif 'P11' in b:
        ct_mann_whitney_u['Age'].loc[b] = 'P2/P3/P11'
    else:# 'P60' in b:
        ct_mann_whitney_u['Age'].loc[b] = 'P60'

In [ ]:
ct_mann_whitney_u

In [ ]:
ctt_mann_whitney_u = ct_mann_whitney_u.reset_index().melt(id_vars=['batch'])[:-8]

In [ ]:
ctt_mann_whitney_u.rename({'value': 'Percentage of cells'}, axis=1, inplace=True)

In [ ]:
ctt_mann_whitney_u['Age'] = None

In [ ]:
ctt_mann_whitney_u = ctt_mann_whitney_u.set_index(['batch'])

In [ ]:
ctt_mann_whitney_u.index

In [ ]:
ctt_mann_whitney_u['Age'] = None

In [ ]:
for b in ctt_mann_whitney_u.index:
    if 'P2' in b:
        ctt_mann_whitney_u['Age'].loc[b] = 'P2/P3/P11'  
    elif 'P3' in b:
        ctt_mann_whitney_u['Age'].loc[b] = 'P2/P3/P11'
    elif 'P11' in b:
        ctt_mann_whitney_u['Age'].loc[b] = 'P2/P3/P11'
    else:# 'P60' in b:
        ctt_mann_whitney_u['Age'].loc[b] = 'P60'

In [ ]:
ctt_mann_whitney_u

In [ ]:
df_undone = {}

for ctype in ctt_mann_whitney_u['Cell_type'].unique():
    for age in ctt_mann_whitney_u['Age'].unique():
        print(ctype, age)
        sdf = ctt_mann_whitney_u[(ctt_mann_whitney_u['Cell_type'] == ctype) & (ctt_mann_whitney_u['Age'] == age)]
        values = list(sdf['Percentage of cells'])
        df_undone[f"{ctype}_{age}"] = values


In [ ]:
df_undone

In [ ]:
from scipy.stats import mannwhitneyu

mann_whitney_u_res = {}

for ctype in ctt_mann_whitney_u['Cell_type'].unique():
    vals_list = []
    for k,v in df_undone.items():
        if ctype in k:
            print(k)
            vals_list.append(df_undone[k])
    f_statistic, p_value = mannwhitneyu(vals_list[0], vals_list[1], method='exact')
    mann_whitney_u_res[ctype] = [f_statistic, p_value]

In [ ]:
mann_whitney_u_res

In [ ]:
mann_whitney_u_res_df = pd.DataFrame(mann_whitney_u_res.items(), columns=['Cell_type', 'Values']).set_index('Cell_type')

In [ ]:
mann_whitney_u_res_df['Values'][0][0]

In [ ]:
mann_whitney_u_res_df['Values'].iloc[[0]]

In [ ]:
mann_whitney_u_res_df['f_statistics'] = None
mann_whitney_u_res_df['p_value'] = None

In [ ]:
for i in range(0, len(mann_whitney_u_res_df['Values'])):
    mann_whitney_u_res_df['f_statistics'].iloc[[i]] = mann_whitney_u_res_df['Values'][i][0]
    mann_whitney_u_res_df['p_value'].iloc[[i]] = mann_whitney_u_res_df['Values'][i][1]

In [ ]:
mann_whitney_u_res_df

In [ ]:
mann_whitney_u_res_df.drop(columns='Values', inplace=True)
mann_whitney_u_res_df

In [ ]:
mann_whitney_u_res_df.to_csv('Mann_Whitney_u_cell_proportions_statistics_all.csv')

# Saving annotated data

In [ ]:
adata.write_h5ad('noDBL_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

# Loading annotated data

In [ ]:
adata = sc.read_h5ad('noDBL_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
adata

In [ ]:
adata.obs['Log10_nGenes'] = np.log10(adata.obs['n_genes_by_counts'])

In [ ]:
vars_plot = ['total_counts', 'Log10_nGenes', 'n_genes_by_counts','pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb']

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(20, 6), sharex=True)

groupby_var = "Cell_type"

# Loop over genes and groupby variables to create violin plots
for i, v in enumerate(vars_plot):
    if i != 1:
        ax = axes[i] if 6 > 1 else axes  # Handle single subplot case
        sc.pl.violin(adata, keys=v, groupby=groupby_var, rotation=90, ax=ax, stripplot=False, palette=my_c, 
                     show=False)

        # Customize y-axis limits for each subplot
        ax.set_ylim(bottom=0)  # Adjust the y-limits as needed for each gene
    else:
        ax = axes[i] if 5 > 1 else axes  # Handle single subplot case
        sc.pl.violin(adata, keys=v, groupby=groupby_var, rotation=90, ax=ax, stripplot=False, palette=my_c, 
                     show=False)

        # Customize y-axis limits for each subplot
        ax.set_ylim(bottom=0, top=5)  # Adjust the y-limits as needed for each gene

# Add a common x-axis label and display the plot
plt.tight_layout()  # Adjust subplot layout to prevent overlap
plt.show()

fig.savefig('violinplots_allcells_metrics.svg', format='svg', dpi=180)

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(20, 5), sharex=True)

groupby_var = "batch"

# Loop over genes and groupby variables to create violin plots
for i, v in enumerate(vars_plot):
    if i != 1:
        ax = axes[i] if 6 > 1 else axes  # Handle single subplot case
        sc.pl.violin(adata, keys=v, groupby=groupby_var, rotation=90, ax=ax, stripplot=False, palette=my_c, 
                     show=False)

        # Customize y-axis limits for each subplot
        ax.set_ylim(bottom=0)  # Adjust the y-limits as needed for each gene
    else:
        ax = axes[i] if 5 > 1 else axes  # Handle single subplot case
        sc.pl.violin(adata, keys=v, groupby=groupby_var, rotation=90, ax=ax, stripplot=False, palette=my_c, 
                     show=False)

        # Customize y-axis limits for each subplot
        ax.set_ylim(bottom=0, top=5)  # Adjust the y-limits as needed for each gene

# Add a common x-axis label and display the plot
plt.tight_layout()  # Adjust subplot layout to prevent overlap
plt.show()

fig.savefig('violinplots_allcells_samples_metrics.svg', format='svg', dpi=180)

## Saving annotated data

In [ ]:
adata.write_h5ad('noDBL_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
adata = sc.read_h5ad('noDBL_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
adata

In [ ]:
sc.set_figure_params(scanpy=True, fontsize=14)

In [ ]:
sc.pl.umap(adata, color=['Cell_type', 'Age'], ncols=4, cmap='magma',wspace=0.7, save='_noDBL_celltypes_age_all.svg')

In [ ]:
adata.uns['log1p']["base"] = None

In [ ]:
sc.tl.filter_rank_genes_groups(adata, groupby='Cell_type', key = "wilcoxon_Cell_type", 
                               key_added='wilcoxon_filtered_leiden_Cell_type',
                               min_in_group_fraction=0.1,max_out_group_fraction=0.5,min_fold_change=0.25)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_filtered_leiden_Cell_type" , groupby="Cell_type", 
                                cmap="YlGnBu", vmin=0, vmax=5, save='_noDBL_filtered_15_all.svg')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, key="wilcoxon_filtered_leiden_Cell_type" , groupby="Cell_type", 
                                cmap="YlGnBu", vmin=0, vmax=5, save='_noDBL_filtered_10_all.svg')

# Gender inference based on Xist-counts and percent_chrY

In [ ]:
adata = sc.read_h5ad('noDBL_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
adata

In [ ]:
adata.obs['Xist-counts']

In [ ]:
adata.obs['percent_chrY']

In [ ]:
adata.obs['Female'] = (adata.obs['Xist-counts'] > 0)

In [ ]:
adata.obs['Female']

In [ ]:
sc.pl.umap(adata, color=['Cell_type', 'Female'], ncols=4, palette= my_c,wspace=0.7, save='Celltypes_gender_all.svg')

In [ ]:
sc.pl.umap(adata, color=['percent_chrY', 'Xist-counts'], ncols=4, palette= my_c,wspace=0.7, vmax=0.2, save='Percent_chrY_Xits_counts.svg')